# Lab 5: ระบบ Vectorless RAG Part I

Vectorless RAG คือแนวทาง RAG ที่ **ไม่ใช้ Dense Embeddings และ Vector Similarity Search** ในการค้นข้อมูล

ระบบอาจค้นข้อมูลด้วยวิธีอื่น เช่น:

- Metadata Filtering
- SQL หรือ Database Query
- Keyword Search หรือ Inverted Index
- Hierarchical Navigation จาก Category/Section
- LLM Reasoning เพื่อเลือกเอกสารจาก Candidate Set

Vectorless ไม่ได้หมายความว่า “ไม่มี Retrieval” แต่หมายถึง Retrieval ไม่ได้อาศัย Vector Space เป็นกลไกหลัก



## เป้าหมาย: 

พัฒนาสร้างระบบแนะนำกระทู้ Pantip ที่เกี่ยวข้อง โดยอ้างอิงข้อมูล [krathu-500
](https://github.com/Pittawat2542/krathu-500) 

โดยอาศัยโครงสร้างข้อมูล (metadata/hierarchy) ที่มีอยู่แล้ว ร่วมกับ LLM ในการ Hierarchical Navigation แทนการค้นหาด้วยความคล้ายของเวกเตอร์

โดยจะแบ่งเป็น 2 Parts ได้แก่

#### Part I: Indexing 

1. **Load** — อ่านข้อมูลจากไฟล์เอกสาร
2. **Analyse** — จัดกลุ่มข้อมูลเป็นหมวดหมู่ แทนการทำ embedding
3. **Structured Index** - แปลงข้อมูลเป็นคลังความรู้แบบ wiki ตาม Open Knowledge Format (OKF) 

#### Part II: Generating 

1. **Query Routing** — สร้าง agent ที่ค้นข้อมูลด้วยการไล่ตาม index/link 
2. **Generation with Sourcing** — ให้ LLM ตอบพร้อมอ้างอิงหมายเลขแหล่งที่มา



## ภาพรวมสถาปัตยกรรม

```mermaid
flowchart TB
  CSV["Load: posts.csv + comments.csv"] --> IG["Knowledge Taxonomy"]
  style CSV fill:#ff9999,stroke:#333
  
  IG --> OKF["OKF wiki bundle"]
  
```

## Step 1: Load — อ่านข้อมูลจากไฟล์เอกสาร

เริ่มต้นจาก download ข้อมูลจาก https://github.com/Pittawat2542/krathu-500

`git clone https://github.com/Pittawat2542/krathu-500.git`


โดยใน Lab นี้ เราจะกำหนดให้ **หนึ่ง `Document` แทนหนึ่งกระทู้** โดยประกอบด้วย
* ชื่อกระทู้
* เนื้อหาต้นกระทู้
* และความคิดเห็นทั้งหมด 

In [3]:
ls krathu-500

README.md                example/                 posts.csv
baseline-model/          labeled/                 requirements.txt
chromedriver*            main.py                  small-dataset-generator/
comments.csv             post-processing/


In [4]:
import pandas as pd

posts_df = pd.read_csv("./krathu-500/post-processing/posts.csv", dtype={"id": "string"})
comments_df = pd.read_csv(
    "./krathu-500/post-processing/comments.csv",
    dtype={
        "comment_id": "string",
        "reply_to": "string",
        "text": "string",
    },
)

In [5]:
# จำกัดจำนวน Posts ที่สนใจ แค่ 100 ข้อมูลเท่านั้น
posts_df = posts_df.head(100)

In [6]:
posts_df.head()

,id,title,url,comment_count,vote_count,published_at
0,38597354,ถึงคนที่มีทุกอย่างอย่างที่ฝันไว้แล้ว..ว่าจริงห...,https://pantip.com/topic/38597354,126,3,2019-02-26 23:06:26+06:42
1,41041562,เด็กจบใหม่กับความเครียดจากครอบครัวและการหางาน,https://pantip.com/topic/41041562,211,0,2021-10-15 01:21:22+06:42
2,41040956,พนักงานออฟฟิศ สิ่งไหนที่เจ้านายทำแล้ว พนักงานร...,https://pantip.com/topic/41040956,275,0,2021-10-14 19:51:54+06:42
3,41031597,มีวิธีรับมืออย่างไรเมื่อเจอกับคนที่ทำงานเก่งมา...,https://pantip.com/topic/41031597,496,15,2021-10-10 09:05:21+06:42
4,30467953,คนขายประกันที่บอกว่าไปเที่ยวเมืองนอก ได้เงินเด...,https://pantip.com/topic/30467953,538,0,2013-05-10 09:59:20+06:42


In [7]:
comments_df.head()

,comment_id,text,collected_at,published_at,reply_to
0,38597354-0,ผมคิดถึงเรื่องนี้มาหลายครั้ง แต่ก็ไม่เชิงว่าค...,2021-10-19 16:42:41.984684,2019-02-26 23:06:26+06:42,<NA>
1,38597354-7817b4e7-6c30-414a-8145-c96864d38409,ซื้อความรักไม่ได้ ถึงเปย์ก็ได้แต่คนไม่จริงใจ,2021-10-19 16:42:42.034824,2019-02-26 23:09:18+06:42,38597354.0
2,38597354-314fe8b7-2ded-4b50-990b-21bded1edbf5,อยากได้ ไม่ได้ทุกข์ พอได้แล้วสุข สุขแล้วเบื่อ\...,2021-10-19 16:42:42.064554,2019-02-27 00:00:36+06:42,38597354.0
3,38597354-4f1ae8a3-3eac-46f3-91de-eca33be44e94,ความสุขมันก็คือการได้รับในสิ่งที่ตัวเองต้องการ...,2021-10-19 16:42:42.096689,2019-02-27 02:31:00+06:42,38597354.0
4,38597354-38e7901d-9402-445d-a235-99d2e772c641,ความสุขของแต่ละคนมันอยู่ที่นิยามของเจ้าตัวครับ,2021-10-19 16:42:42.125382,2019-02-27 04:34:28+06:42,38597354.0


In [8]:
comments_df["post_id"] = comments_df["comment_id"].str.extract(r"^(\d+)-", expand=False)

In [9]:
print(f"posts.csv: {len(posts_df):,} records")
print(f"comments.csv: {len(comments_df):,} records")

posts.csv: 100 records
comments.csv: 63,867 records


## Step 2: Analyse — จัดกลุ่มข้อมูลเป็นหมวดหมู่

โดยในแล๊บนี้ จะสร้าง Taxonomy ของข้อมูล จาก **post titles เท่านั้น** จากนั้นจึง classify ทุก title เข้า taxonomy แบบ batch ตามลำดับ


#### 2.1 Formatting Data

In [13]:
from pydantic import BaseModel, Field

class Comment(BaseModel):
    comment_id: str
    text: str = ""
    published_at: str | None = None
    collected_at: str | None = None
    reply_to: str | None = None


class PostThread(BaseModel):
    post_id: str
    title: str
    url: str | None = None
    vote_count: int = 0
    comment_count: int = 0
    published_at: str | None = None
    body: str = ""
    comments: list[Comment] = Field(default_factory=list)

In [23]:
grouped = {str(k): g for k, g in comments_df.groupby("post_id", sort=False)}

threads = []
for row in posts_df.to_dict("records"):
    post_id = str(row["id"])
    group = grouped.get(post_id, comments_df.iloc[0:0])
    body_rows = group[group["comment_id"] == f"{post_id}-0"]
    if body_rows.empty:
        continue

    body = body_rows.iloc[0]["text"]
    if body is None or pd.isna(body) or str(body).strip()=="":
        continue

    body = str(body).strip()

    discussion = group[group["comment_id"] != f"{post_id}-0"].copy()
    comments = []
    for item in discussion.to_dict("records"):
        text = item["text"]
        if text is None or pd.isna(text) or str(text).strip()=="":
            continue

        comments.append(Comment(
            comment_id=str(item["comment_id"]),
            text=item["text"],
            published_at=item["published_at"],
            collected_at=item["collected_at"],
            reply_to=item["reply_to"],  
        ))
        
    threads.append(PostThread(
        post_id=post_id,
        title=str(row["title"]),
        url=row.get("url"),
        vote_count=row.get("vote_count"),
        comment_count=row.get("comment_count"),
        published_at=row.get("published_at"),
        body=body,
        comments=comments,
    ))

In [26]:
len(threads[0].comments)

125

#### 2.2 Create Taxonomy

In [27]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
GEMINI_KEY = os.getenv("GEMINI_KEY")
MODEL_NAME = "gemini-3.5-flash-lite"

llm = ChatGoogleGenerativeAI(api_key=GEMINI_KEY, model=MODEL_NAME)

In [28]:
class Subcategory(BaseModel):
    id: str = Field(description="lowercase kebab-case id")
    title: str
    description: str

class Category(BaseModel):
    id: str = Field(description="lowercase kebab-case id")
    title: str
    description: str
    subcategories: list[Subcategory] = Field(default_factory=list)

class Taxonomy(BaseModel):
    categories: list[Category]

taxonomy_llm = llm.with_structured_output(Taxonomy)

In [38]:
titles = [f"* {t.title}" for t in threads]

In [39]:
titles[0:10]

['* ถึงคนที่มีทุกอย่างอย่างที่ฝันไว้แล้ว..ว่าจริงหรือ ที่เงินมันซื้อความสุขได้',
 '* เด็กจบใหม่กับความเครียดจากครอบครัวและการหางาน',
 '* พนักงานออฟฟิศ สิ่งไหนที่เจ้านายทำแล้ว พนักงานรู้สึกแย่สุด',
 '* มีวิธีรับมืออย่างไรเมื่อเจอกับคนที่ทำงานเก่งมากๆ แต่ไม่ค่อยเข้ากลุ่มเข้าสังคมอะไรเลย',
 '* คนขายประกันที่บอกว่าไปเที่ยวเมืองนอก ได้เงินเดือนเป็นแสนมันจริงหรือค่ะ',
 '* หมอไม่ได้รวยอย่างที่คิด งานหนักกว่าอาขีพอื่น2เท่า พ่อแม่อย่าบังคับลูกเรียนเลยครับ',
 '* มีใครเป็นเจ้าของกิจการอายุไม่มากบ้างครับ รายได้ต่อเดือนประมาณเท่าไรกันบ้างครับ',
 '* รู้สึกหมดไฟ ไม่มีความสุขในการทำงาน อยากลาออก',
 '* 10 คำขอพิลึกจากลูกค้าที่พนักงานโรงแรมปวดกบาลที่สุด!?',
 '* สรุปแล้ว.ปัญหาแบงค์พาณิชย์หลอกเงินฝากเป็นประกันชีวิตเอาเป้าเข้าตัวเอง บ.ประกันได้เงิน คนมันโง่เอง.แบงค์ชาติไม่เกี่ยวหรือ']

In [48]:
import json

title_list = "\n".join(titles)
prompt = f"""
Design navigation for an OKF wiki. Create 6-10 broad categories and useful subcategories using ONLY the post titles below. 
IDs must be unique lowercase kebab-case. 
Include category id 'other' with subcategory id 'uncategorized'.
Do not infer facts from post bodies because you cannot see them.

TITLES:\n{title_list}"""

In [49]:
taxonomy_response = taxonomy_llm.invoke(prompt)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [50]:
!uv pip install -q textcase

In [51]:
import textcase

def normalize_taxonomy(taxonomy: Taxonomy):
    """Make model-generated IDs path-safe and guarantee a deterministic fallback."""
    normalized_categories = []
    used_categories: set[str] = set()
    
    for category_number, category in enumerate(taxonomy.categories, start=1):
        category_id = textcase.kebab(category.id.lower()) or f"category-{category_number}"
        
        # If category is duplicated
        if category_id in used_categories:
            category_id = f"{category_id}-{category_number}"

        used_categories.add(category_id)

        
        normalized_subcategories = []
        used_subcategories: set[str] = set()
        for sub_number, sub in enumerate(category.subcategories, start=1):
            sub_id = textcase.kebab(sub.id.lower()) or f"{category_id}-subcategory-{sub_number}"
            if sub_id in used_subcategories:
                sub_id = f"{sub_id}-{sub_number}"
                
            used_subcategories.add(sub_id)
            normalized_subcategories.append(Subcategory(id=sub_id, title=sub.title, description=sub.description))

            
        normalized_categories.append(Category(
            id=category_id,
            title=category.title,
            description=category.description,
            subcategories=normalized_subcategories,
        ))
        
    if "other" not in used_categories:
        normalized_categories.append(Category(
            id="other",
            title="Other",
            description="Posts that do not fit the generated taxonomy with confidence.",
            subcategories=[Subcategory(
                id="uncategorized",
                title="Uncategorized",
                description="Fallback for uncertain title classifications.",
            )],
        ))
    else:
        other = next(c for c in normalized_categories if c.id == "other")
        if "uncategorized" not in {s.id for s in other.subcategories}:
            other.subcategories.append(Subcategory(
                id="uncategorized",
                title="Uncategorized",
                description="Fallback for uncertain title classifications.",
            ))
            
    return Taxonomy(categories=normalized_categories)

taxonomy = normalize_taxonomy(taxonomy_response)

In [56]:
for category in taxonomy.categories:
    print(f"* {category.id}")
    for subcategory in category.subcategories:
        print(f"\t* {subcategory.id}")

* employment-and-career
	* job-search-and-new-graduates
	* workplace-relationships-and-culture
	* resignation-and-career-change
	* government-and-corporate-jobs
* finance-and-wealth
	* salary-and-income
	* saving-and-debt
	* business-and-investments
* retirement-and-aging
	* retirement-planning
	* senior-employment
* lifestyle-and-well-being
	* happiness-and-mental-health
	* personal-experiences
* society-and-family
	* family-and-culture
	* social-issues
* other
	* uncategorized


#### 2.3 Classify Posts

In [57]:
categories = {c.id for c in taxonomy.categories}
subcategories = {c.id: {s.id for s in c.subcategories} for c in taxonomy.categories}

In [63]:
compact_taxonomy = [
    {
        "id": c.id,
        "title": c.title,
        "subcategories": [{"id": s.id, "title": s.title} for s in c.subcategories],
    }
    for c in taxonomy.categories
]

# compact_taxonomy

In [64]:
class PostClassification(BaseModel):
    post_id: str
    category_id: str
    subcategory_id: str

class ClassificationBatch(BaseModel):
    items: list[PostClassification]
    
classify_llm = llm.with_structured_output(ClassificationBatch)

In [75]:
from tqdm import tqdm

classifications = {}

BATCH_SIZE = 50
for start in tqdm(range(0, len(threads), BATCH_SIZE)):
    batch = threads[start:start + BATCH_SIZE]
    title_rows = [{"post_id": t.post_id, "title": t.title} for t in batch]
    
    prompt = "Classify every post title into exactly one category and subcategory from the given taxonomy. Use 'other/uncategorized' when uncertain. Return every post_id once.\n\n"
    prompt += "===========\n"
    prompt += f"TAXONOMY:\n{json.dumps(compact_taxonomy, ensure_ascii=False)}\n"
    prompt += "===========\n"
    prompt += f"TITLES:\n{json.dumps(title_rows, ensure_ascii=False)}\n"
    
    
    result = classify_llm.invoke(prompt)
    for item in result.items:
        category = item.category_id if item.category_id in categories else "other"
        
        allowed_subcategories = subcategories.get(category, set())
        subcategory = item.subcategory_id if item.subcategory_id in allowed_subcategories else "uncategorized"
        
        classifications[item.post_id] = {
            "category_id": category,
            "subcategory_id": subcategory,
        }

# set other:uncategorized for missing posts
for thread in threads:
    if thread.post_id not in classifications:
        classifications[thread.post_id] = {"category_id": "other", "subcategory_id": "uncategorized"},


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:13<00:00,  6.97s/it]


In [77]:
len(classifications)

100

## Step 3: Structured Index — แปลงข้อมูลเป็นคลังความรู้แบบ wiki

โครงสร้างผลลัพธ์:

```text
pantip-wiki/
├── index.md
├── log.md
├── references/
├── topics/<category>/<subcategory>/index.md
└── posts/<post-id>/
    ├── index.md
    ├── post.md
    ├── overview.md
    ├── discussion/index.md + <theme>.md
    └── comments/index.md + part-*.md
```

- หน้าที่เป็นข้อมูลดั้งเดิม ใช้ `status: stable`
- หน้าที่ LLM สร้าง ใช้ `status: draft` + มี `sources` ชี้กลับไป raw pages

see: https://github.com/GoogleCloudPlatform/open-knowledge-format/tree/main


#### 3.1 Build Wiki-like Structure

In [80]:
from pathlib import Path

ROOT_DIR = "./pantip-wiki"
root = Path(ROOT_DIR)

In [166]:
import yaml
from datetime import datetime, timezone, timedelta
def now_iso():
    return datetime.now(timezone(timedelta(hours=7))).replace(microsecond=0).date().isoformat()
    
def write_markdown(path: Path, metadata: dict, body: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    body = body.strip() + "\n"

    # OKF reserves index.md for navigation and forbids its frontmatter.
    if path.name == "index.md":
        path.write_text(body, encoding="utf-8")
        return

    frontmatter = yaml.safe_dump(metadata, allow_unicode=True, sort_keys=False).strip()
    path.write_text(f"---\n{frontmatter}\n---\n\n{body.strip()}\n", encoding="utf-8")

def rel_link(from_file: Path, to_file: Path, label: str):
    relative = os.path.relpath(to_file, start=from_file.parent).replace(os.sep, "/")
    return f"[{label}]({relative})"
    
def write_raw_post_pages(root: Path, thread: PostThread):
    post_dir = root / "posts" / thread.post_id
    raw_post = post_dir / "post.md"
    
    post_body = ""
    post_body += f"# {thread.title}\n\n"
    post_body += f"{thread.body}\n\n"
    post_body += f"Original URL: {thread.url}\n"

    metadata = {
        "type": "Post",
        "title": thread.title,
        "description": "Raw post content and corpus metadata",
        "status": "stable",
        "updated": now_iso(),
        "sources": [thread.url] if thread.url else None,
        "tags": ["raw-source", "post"],
        
        "post_id": thread.post_id,
        "published_at": thread.published_at,
    }
    
    write_markdown(raw_post, metadata, post_body)

    COMMENT_PAGE_SIZE = 50
    comment_pages = []
    for page_number, start in enumerate(range(0, len(thread.comments), COMMENT_PAGE_SIZE), start=1):
        batch = thread.comments[start:start + COMMENT_PAGE_SIZE]
        page = post_dir / "comments" / f"part-{page_number:03d}.md"
        blocks = []
        for comment in batch:
            reply = f" · reply_to `{comment.reply_to}`" if comment.reply_to else ""
            blocks.append(f"## `{comment.comment_id}`{reply}\n\n{comment.text}")

        comment_metadata = {
            "type": "Comments",
            "title": f"Comments for {thread.title} — part {page_number}",
            "description": "Comments from the source corp",
            "status": "stable",
            "updated": now_iso(),
            "sources": [rel_link(page, raw_post, thread.title)],
            "tags": ["raw-source", "comments"],
            
            "post_id": thread.post_id,
            "published_at": thread.published_at,
        }
        
        write_markdown(page, comment_metadata, "\n\n".join(blocks))
        comment_pages.append(page)
        
    return comment_pages

In [167]:
comment_page_index = {}
for thread in threads:
    comment_pages = write_raw_post_pages(root, thread)
    comment_page_index[thread.post_id] = comment_pages

#### 3.2 Analyse Post Content into Topics

In [161]:
MAX_COMMENT_CHARS = 45000
def thread_prompt_payload(thread: PostThread, max_chars: int = MAX_COMMENT_CHARS):
    chunks = [
        "\n".join([
            f"POST ID: {thread.post_id}",
            f"TITLE: {thread.title}",
            f"BODY:\n{thread.body}",
        ])
    ]
    
    chunk_length = len(chunks[0])
    for comment in thread.comments:
        block = f"\nCOMMENT {comment.comment_id} reply_to={comment.reply_to}\n{comment.text}"
        if chunk_length + len(block) > max_chars:
            break
            
        chunks.append(block)
        chunk_length += len(block)
        
    return "\n".join(chunks)

# print(thread_prompt_payload(threads[0]))

class ThemeSection(BaseModel):
    title: str
    summary: str
    key_points: list[str]
    source_comment_ids: list[str]

class ThreadAnalysis(BaseModel):
    overview: str
    themes: list[ThemeSection] = Field(default_factory=list)


analysis_llm = llm.with_structured_output(ThreadAnalysis)

In [122]:
def analyze_and_write_thread(root: Path, thread: PostThread, raw_comment_pages: list[Path]):
    post_dir = root / "posts" / thread.post_id
    raw_post = post_dir / "post.md"
    valid_comment_ids = {c.comment_id for c in thread.comments}

    discussion_index = post_dir / "discussion" / "index.md"
    # Skip the entire analysis if the discussion index already exists
    if discussion_index.exists():
        return

    prompt = ""
    prompt += "Analyze this discussion for a wiki.\n"
    prompt += "* Produce a factual overview and 3-6 non-overlapping themes.\n"
    prompt += "* Cite only COMMENT IDs that appear in the input.\n"
    prompt += "* Do not invent claims or IDs.\n"
    prompt += "* Keep uncertainty and disagreement visible.\n"
    prompt += "* The answer MUST be written in THAI.\n\n"
    prompt += "================\n\n"
    prompt += f"{thread_prompt_payload(thread)}"
    
    analysis = analysis_llm.invoke(prompt)

    overview = post_dir / "overview.md"
    source_links = [rel_link(overview, raw_post, "Raw post")]
    source_links += [rel_link(overview, p, p.stem) for p in raw_comment_pages]

    metadata = {
        "type": "Overview",
        "title": f"Overview: {thread.title}",
        "description": "LLM-generated overview grounded in the post and comments",
        "status": "draft",
        "updated": now_iso(),
        "sources": source_links,
        "tags": ["overview"],
        
        "post_id": thread.post_id,
    }

    write_markdown(overview, metadata, f"# Overview\n\n{analysis.overview}")

    discussion_index = post_dir / "discussion" / "index.md"
    theme_links = []
    used_slugs = set()
    for number, theme in enumerate(analysis.themes, start=1):

        slug = textcase.kebab(theme.title.lower()) or f"theme-{number}"
        
        # If base is duplicated
        if slug in used_slugs:
            slug = f"{slug}-{number}"
            
        used_slugs.add(slug)

        
        theme_page = discussion_index.parent / f"{slug}.md"
        ids = [cid for cid in theme.source_comment_ids if cid in valid_comment_ids]
        points = "\n".join(f"- {point}" for point in theme.key_points)
        citations = ", ".join(f"`{cid}`" for cid in ids) or "_ไม่มี comment citation_"

        metadata = {
            "type": "Discussion",
            "title": theme.title,
            "description": theme.summary,
            "status": "draft",
            "updated": now_iso(),
            "sources": [rel_link(theme_page, raw_post, "Raw post")] + [rel_link(theme_page, p, p.stem) for p in raw_comment_pages],
            "tags": ["discussion-theme"],
            
            "post_id": thread.post_id,
            "source_comment_ids": ids,
        }

        post_body = ""
        post_body += f"# {theme.title}\n\n{theme.summary}\n\n## Key points\n\n{points}\n\n"
        post_body += f"## Source comment IDs\n\n{citations}"
        
        write_markdown(theme_page, metadata, post_body)
        
        theme_links.append(f"- [{theme.title}]({theme_page.name}) — {theme.summary}")

    if theme_links:

        metadata = {
            "title": f"Discussion themes: {thread.title}",
            "description": "Index of synthesized discussion themes",
            "status": "draft",
            "updated": now_iso(),
            "sources": [rel_link(discussion_index, raw_post, "Raw post")],
            "tags": ["index"]
        }
        
        write_markdown(discussion_index, metadata, "# Discussion themes\n\n" + "\n".join(theme_links))


from tqdm import tqdm

for thread in tqdm(threads, total=len(threads)):
    comment_pages = comment_page_index[thread.post_id]
    try:
        # analyze_and_write_thread(root, thread, comment_pages)
    except Exception as e:
        print(f" Error Post:", thread.post_id)
        print(e)
        print()


  0%|                                                                                                                                  | 0/100 [00:00<?, ?it/s]Gemini produced an empty response. Continuing with empty message
Feedback: block_reason=<BlockedReason.PROHIBITED_CONTENT: 'PROHIBITED_CONTENT'> block_reason_message=None safety_ratings=None
  9%|██████████▉                                                                                                               | 9/100 [00:01<00:15,  5.98it/s]

 Error Post: 32247592
Invalid json output: 
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 



 36%|███████████████████████████████████████████▌                                                                             | 36/100 [03:28<09:28,  8.88s/it]Gemini produced an empty response. Continuing with empty message
Feedback: block_reason=<BlockedReason.PROHIBITED_CONTENT: 'PROHIBITED_CONTENT'> block_reason_message=None safety_ratings=None
 37%|████████████████████████████████████████████▊                                                                            | 37/100 [03:30<07:10,  6.84s/it]

 Error Post: 34612385
Invalid json output: 
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 



100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [11:23<00:00,  6.84s/it]


In [124]:
# Feedback: block_reason=<BlockedReason.PROHIBITED_CONTENT: 'PROHIBITED_CONTENT'> block_reason_message=None safety_ratings=None
# Error Post: 32247592
# Invalid json output: 
# For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

# Feedback: block_reason=<BlockedReason.PROHIBITED_CONTENT: 'PROHIBITED_CONTENT'> block_reason_message=None safety_ratings=None
# Error Post: 34612385
# Invalid json output: 
# For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

# 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [11:23<00:00,  6.84s/it]

#### 3.3 Build Index

In [162]:
def category_lookup(taxonomy: Taxonomy):
    result = {}
    for category in taxonomy.categories:
        for subcategory in category.subcategories:
            result[(category.id, subcategory.id)] = (category.title, subcategory.title)
    return result


lookup = category_lookup(taxonomy)

In [168]:
# Add references
refs = root / "references"
write_markdown(
    refs / "posts-corpus.md",
    {
        "type": "Corpus",
        "title": "Posts corpus",
        "description": "Description of posts.csv",
        "status": "stable",
        "updated": now_iso(),
        "sources": [],
        "tags": [],
    },
    "# posts.csv\n\nSource columns: id, title, url, comment_count, vote_count, published_at.",
)

write_markdown(
    refs / "comments-corpus.md",
    {
        "type": "Corpus",
        "title": "Comments corpus",
        "description": "Description of comments.csv",
        "status": "stable",
        "updated": now_iso(),
        "sources": [],
        "tags": [],
    },
    "# comments.csv\n\nThe post id is derived from the numeric prefix of comment_id; suffix 0 is the post body.",
)

In [169]:
# create post index + comment index
by_subcategory = {}
all_post_links = []
for thread in threads:
    post_dir = root / "posts" / thread.post_id
    post_index = post_dir / "index.md"
    raw_post = post_dir / "post.md"
    
    links = [f"- {rel_link(post_index, raw_post, 'Raw post')} — source content and metadata"]
    if (post_dir / "overview.md").exists():
        links.append(f"- {rel_link(post_index, post_dir / 'overview.md', 'Overview')} — synthesized overview")
        
    if (post_dir / "discussion" / "index.md").exists():
        links.append(f"- {rel_link(post_index, post_dir / 'discussion' / 'index.md', 'Discussion themes')}")
        
    comment_parts = sorted((post_dir / "comments").glob("part-*.md"))
    
    comments_index = post_dir / "comments" / "index.md"
    part_links = [f"- {rel_link(comments_index, p, p.stem)}" for p in comment_parts]
    write_markdown(
        comments_index,
        {
            "title": "Comment pages: {thread.title}",
            "description": "Index of raw comment pages",
            "status": "stable",
            "updated": now_iso(),
            "sources": [],
            "tags": [],
        },
        "# Comment pages\n\n" + "\n".join(part_links),
    )
    
    links.append(f"- {rel_link(post_index, comments_index, 'Raw comments')}")
    write_markdown(
        post_index,
        {
            "title": thread.title,
            "description": f"Navigation hub for post {thread.post_id}",
            "status": "stable",
            "updated": now_iso(),
            "sources": [],
            "tags": ["post-index"],

            "post_id": thread.post_id,
        },
        f"# {thread.title}\n\n" + "\n".join(links),
    )
    
    cls = classifications[thread.post_id]
    key = (cls["category_id"], cls["subcategory_id"])
    by_subcategory.setdefault(key, []).append(thread)
    all_post_links.append(f"- [{thread.title}]({thread.post_id}/index.md) — post {thread.post_id}")

In [170]:
posts_index = root / "posts" / "index.md"
write_markdown(
    posts_index,
    {
        "title": "All posts",
        "description": f"Alphabetical-style corpus navigation",
        "status": "stable",
        "updated": now_iso(),
        "sources": [],
        "tags": [],
    },
    "# All posts\n\n" + "\n".join(all_post_links),
)

In [171]:
# create categories page
category_links = []
for category in taxonomy.categories:
    category_index = root / "topics" / category.id / "index.md"
    sub_links = []
    for sub in category.subcategories:
        sub_index = root / "topics" / category.id / sub.id / "index.md"
        post_links = []
        for thread in by_subcategory.get((category.id, sub.id), []):
            target = root / "posts" / thread.post_id / "index.md"
            post_links.append(f"- {rel_link(sub_index, target, thread.title)}")
            
        body = f"# {sub.title}\n\n{sub.description}\n\n" + ("\n".join(post_links) or "_No posts assigned._")
        write_markdown(
            sub_index,
            {
                "title": sub.title,
                "description": sub.description,
                "status": "draft",
                "updated": now_iso(),
                "sources": [],
                "tags": [],
            },
            body,
        )
        sub_links.append(f"- {rel_link(category_index, sub_index, sub.title)} — {sub.description}")
        
    write_markdown(
        category_index,
        {
            "title": category.title,
            "description": category.description,
            "status": "draft",
            "updated": now_iso(),
            "sources": [],
            "tags": [],
        },
        f"# {category.title}\n\n{category.description}\n\n" + "\n".join(sub_links),
    )
    category_links.append(f"- [{category.title}](topics/{category.id}/index.md) — {category.description}")

In [172]:
write_markdown(
    root / "index.md",
    {
        "okf_version": "0.2",
        "title": "Posts and discussions knowledge base",
        "description": "Wiki-style OKF bundle generated from the supplied CSV corpus",
        "status": "draft",
        "updated": now_iso(),
        "sources": [],
        "tags": [],
    },
    "# Knowledge base\n\n## Browse by topic\n\n" + "\n".join(category_links)
    + "\n\n## Other entry points\n\n- [All posts](posts/index.md)\n- [Corpus references](references/posts-corpus.md)\n- [Build log](log.md)",
)
write_markdown(
    root / "log.md",
    {
        "title": "Build log",
        "description": "Ingestion summary",
        "status": "stable",
        "updated": now_iso(),
        "sources": [],
        "tags": [],
    },
    f"# Build log\n\n- Built at: {now_iso()}\n- Posts processed: {len(threads)}\n",
)

#### 3.4 Validate

In [173]:
# !uv tool install okflint

In [174]:
# https://github.com/mattdav/okflint

In [203]:
!okflint validate --manifest ./okf-base.yaml ./pantip-wiki/

⚠️ [L001] pantip-wiki/posts/40960232/comments/part-004.md — broken wikilink: [[พ่อแม่ เลี้ยงดูลูกมา]]
⚠️ [L001] pantip-wiki/posts/40960232/comments/.ipynb_checkpoints/part-004-checkpoint.md — broken wikilink: [[พ่อแม่ เลี้ยงดูลูกมา]]
⚠️ [L001] pantip-wiki/posts/32568194/comments/part-004.md — broken wikilink: [[-----CAT HAS TROPHY------]]

0 error(s), 3 warning(s).


## More Information

ดูข้อมูลเพิ่มเติมเกี่ยวกับ Vectorless RAG ได้ที่
* [PageIndex](https://github.com/VectifyAI/PageIndex)
* [LLM-wiki](https://gist.github.com/karpathy/442a6bf555914893e9891c11519de94f) หรือ [https://llm-wiki.net](https://llm-wiki.net/#install)
* [บันทึกความเข้าใจเกี่ยวกับ LLM Wiki และใช้งานร่วมกับ Ollama](https://www.somkiat.cc/learn-llm-wiki/)